In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
file_path = Path("../Dataset/sms-call-internet-mi-2013-11-01.csv")

df_raw = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Shape:", df_raw.shape)

Dataset loaded successfully
Shape: (1891928, 8)


In [4]:
print("\nRaw column names:")
print(df_raw.columns.tolist())

print("\nData types:")
print(df_raw.dtypes)

print("\nFirst 5 rows:")
display(df_raw.head())

print("Missing values per column:")
display(df_raw.isnull().sum())


Raw column names:
['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin', 'callout', 'internet']

Data types:
datetime           str
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object

First 5 rows:


,datetime,CellID,countrycode,smsin,smsout,callin,callout,internet
0,2013-11-01 00:00:00,1,0,0.3521,NaN,NaN,0.0273,NaN
1,2013-11-01 00:00:00,1,33,NaN,NaN,NaN,NaN,0.0261
2,2013-11-01 00:00:00,1,39,1.7322,1.1047,0.5919,0.4020,57.7729
3,2013-11-01 00:00:00,2,0,0.3581,NaN,NaN,0.0273,NaN
4,2013-11-01 00:00:00,2,33,NaN,NaN,NaN,NaN,0.0274


Missing values per column:


datetime             0
CellID               0
countrycode          0
smsin          1086153
smsout         1422446
callin         1407781
callout        1037413
internet       1087074
dtype: int64

In [5]:
column_mapping = {
    "datetime": "timestamp",
    "CellID": "grid_id",
    "countrycode": "country_code",
    "smsin": "sms_in",
    "smsout": "sms_out",
    "callin": "call_in",
    "callout": "call_out",
    "internet": "internet_activity"
}

df = df_raw.rename(columns=column_mapping).copy()

print("Canonical columns:")
print(df.columns.tolist())

required_columns = [
    "timestamp",
    "grid_id",
    "country_code",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

Canonical columns:
['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity']
All required columns are present.


In [6]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("Timestamp dtype:", df["timestamp"].dtype)

print("\nInvalid timestamps:", df["timestamp"].isna().sum())

numeric_columns = [
    "grid_id",
    "country_code",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print(df.dtypes)

timestamps = (
    df["timestamp"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Number of distinct timestamps:", len(timestamps))

print("\nTimestamps:")
display(timestamps)

time_difference = timestamps.diff().dropna()

print("Unique time differences:")
print(time_difference.unique())

hourly_cadence = (
    len(timestamps) == 24
    and (time_difference == pd.Timedelta(hours=1)).all()
)

print("\nHourly cadence valid:", hourly_cadence)

Timestamp dtype: datetime64[us]

Invalid timestamps: 0
timestamp            datetime64[us]
grid_id                       int64
country_code                  int64
sms_in                      float64
sms_out                     float64
call_in                     float64
call_out                    float64
internet_activity           float64
dtype: object
Number of distinct timestamps: 24

Timestamps:


0    2013-11-01 00:00:00
1    2013-11-01 01:00:00
2    2013-11-01 02:00:00
3    2013-11-01 03:00:00
4    2013-11-01 04:00:00
5    2013-11-01 05:00:00
6    2013-11-01 06:00:00
7    2013-11-01 07:00:00
8    2013-11-01 08:00:00
9    2013-11-01 09:00:00
10   2013-11-01 10:00:00
11   2013-11-01 11:00:00
12   2013-11-01 12:00:00
13   2013-11-01 13:00:00
14   2013-11-01 14:00:00
15   2013-11-01 15:00:00
16   2013-11-01 16:00:00
17   2013-11-01 17:00:00
18   2013-11-01 18:00:00
19   2013-11-01 19:00:00
20   2013-11-01 20:00:00
21   2013-11-01 21:00:00
22   2013-11-01 22:00:00
23   2013-11-01 23:00:00
Name: timestamp, dtype: datetime64[us]

Unique time differences:
<TimedeltaArray>
['0 days 01:00:00']
Length: 1, dtype: timedelta64[us]

Hourly cadence valid: True


In [7]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()

display(
    df[
        [
            "timestamp",
            "date",
            "hour",
            "day_of_week"
        ]
    ].head()
)

,timestamp,date,hour,day_of_week
0,2013-11-01,2013-11-01,0,Friday
1,2013-11-01,2013-11-01,0,Friday
2,2013-11-01,2013-11-01,0,Friday
3,2013-11-01,2013-11-01,0,Friday
4,2013-11-01,2013-11-01,0,Friday


In [8]:
missing_grid_id = df["grid_id"].isna().sum()
missing_timestamp = df["timestamp"].isna().sum()

print("Missing grid_id:", missing_grid_id)
print("Missing timestamp:", missing_timestamp)

activity_columns = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

null_activity_counts = df[activity_columns].isna().sum()

print("Blank activity measures:")
display(null_activity_counts)

duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

negative_counts = {}

for col in activity_columns:
    negative_counts[col] = (df[col] < 0).sum()

negative_counts = pd.Series(negative_counts)

print("Negative activity values:")
display(negative_counts)

print("Unique grids:", df["grid_id"].nunique())

print("Minimum grid_id:", df["grid_id"].min())
print("Maximum grid_id:", df["grid_id"].max())

invalid_grid_ids = df[
    ~df["grid_id"].between(1, 10000)
]

print("Invalid grid IDs:", len(invalid_grid_ids))

Missing grid_id: 0
Missing timestamp: 0
Blank activity measures:


sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
dtype: int64

Exact duplicate rows: 0
Negative activity values:


sms_in               0
sms_out              0
call_in              0
call_out             0
internet_activity    0
dtype: int64

Unique grids: 10000
Minimum grid_id: 1
Maximum grid_id: 10000
Invalid grid IDs: 0


In [9]:
country_codes = (
    df["country_code"]
    .dropna()
    .value_counts()
    .sort_index()
)

print("Country-code categories:")
display(country_codes)

grid_hour_counts = (
    df.groupby(
        ["timestamp", "grid_id"]
    )
    .size()
    .reset_index(name="row_count")
)

print("Grid/hour combinations:", len(grid_hour_counts))

print("\nExamples with multiple country-code rows:")
display(
    grid_hour_counts[
        grid_hour_counts["row_count"] > 1
    ].head(10)
)

example = (
    grid_hour_counts[
        grid_hour_counts["row_count"] > 1
    ]
    .iloc[0]
)

example_timestamp = example["timestamp"]
example_grid = example["grid_id"]

display(
    df[
        (df["timestamp"] == example_timestamp) &
        (df["grid_id"] == example_grid)
    ]
)

Country-code categories:


country_code
0        231307
1         42777
7         27509
20        78182
27         1260
          ...  
18768        13
68572        28
88235         9
88239      4259
97259        51
Name: count, Length: 246, dtype: int64

Grid/hour combinations: 240000

Examples with multiple country-code rows:


,timestamp,grid_id,row_count
0,2013-11-01,1,3
1,2013-11-01,2,3
2,2013-11-01,3,3
3,2013-11-01,4,3
4,2013-11-01,5,3
5,2013-11-01,6,3
6,2013-11-01,7,3
7,2013-11-01,8,3
8,2013-11-01,9,3
9,2013-11-01,10,3


,timestamp,grid_id,country_code,sms_in,sms_out,call_in,call_out,internet_activity,date,hour,day_of_week
0,2013-11-01,1,0,0.3521,NaN,NaN,0.0273,NaN,2013-11-01,0,Friday
1,2013-11-01,1,33,NaN,NaN,NaN,NaN,0.0261,2013-11-01,0,Friday
2,2013-11-01,1,39,1.7322,1.1047,0.5919,0.4020,57.7729,2013-11-01,0,Friday


In [10]:
df["total_sms"] = (
    df["sms_in"].fillna(0)
    + df["sms_out"].fillna(0)
)

df["total_calls"] = (
    df["call_in"].fillna(0)
    + df["call_out"].fillna(0)
)

df["total_activity"] = (
    df["total_sms"]
    + df["total_calls"]
    + df["internet_activity"].fillna(0)
)

display(
    df[
        [
            "timestamp",
            "grid_id",
            "country_code",
            "total_sms",
            "total_calls",
            "internet_activity",
            "total_activity"
        ]
    ].head()
)

,timestamp,grid_id,country_code,total_sms,total_calls,internet_activity,total_activity
0,2013-11-01,1,0,0.3521,0.0273,NaN,0.3794
1,2013-11-01,1,33,0.0000,0.0000,0.0261,0.0261
2,2013-11-01,1,39,2.8369,0.9939,57.7729,61.6037
3,2013-11-01,2,0,0.3581,0.0273,NaN,0.3854
4,2013-11-01,2,33,0.0000,0.0000,0.0274,0.0274


In [12]:
hourly_activity = (
    df.groupby("timestamp")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)

print("Busiest hourly window:")
display(hourly_activity.head(10))

grid_activity = (
    df.groupby("grid_id")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 busiest grids:")
display(grid_activity.head(10))

Busiest hourly window:


timestamp
2013-11-01 11:00:00    5.185704e+06
2013-11-01 18:00:00    5.135660e+06
2013-11-01 17:00:00    5.108601e+06
2013-11-01 12:00:00    5.038444e+06
2013-11-01 10:00:00    4.926999e+06
2013-11-01 16:00:00    4.916011e+06
2013-11-01 19:00:00    4.912902e+06
2013-11-01 15:00:00    4.808330e+06
2013-11-01 14:00:00    4.774330e+06
2013-11-01 20:00:00    4.700756e+06
Name: total_activity, dtype: float64

Top 10 busiest grids:


grid_id
5161    274800.2956
5059    197968.3438
5758    143489.2707
5061    137435.4867
5955    135844.9556
5262    131322.8759
6064    126818.9053
5162    124664.3217
4857    121991.8232
4856    114005.3370
Name: total_activity, dtype: float64

In [13]:
profiling = {
    "row_count": len(df),
    "column_count": len(df.columns),
    "unique_grids": df["grid_id"].nunique(),
    "unique_timestamps": df["timestamp"].nunique(),
    "min_timestamp": df["timestamp"].min(),
    "max_timestamp": df["timestamp"].max(),
    "country_code_categories": df["country_code"].nunique(),
    "exact_duplicates": df.duplicated().sum(),
    "missing_grid_id": df["grid_id"].isna().sum(),
    "missing_timestamp": df["timestamp"].isna().sum(),
    "hourly_cadence_valid": hourly_cadence,
    "busiest_hour": hourly_activity.idxmax(),
    "busiest_hour_activity": hourly_activity.max(),
    "busiest_grid": grid_activity.idxmax(),
    "busiest_grid_activity": grid_activity.max()
}

profiling_df = pd.DataFrame(
    profiling.items(),
    columns=["Metric", "Value"]
)

display(profiling_df)

,Metric,Value
0,row_count,1891928
1,column_count,14
2,unique_grids,10000
3,unique_timestamps,24
4,min_timestamp,2013-11-01 00:00:00
5,max_timestamp,2013-11-01 23:00:00
6,country_code_categories,246
7,exact_duplicates,0
8,missing_grid_id,0
9,missing_timestamp,0


In [16]:
assert df["timestamp"].nunique() == 24, \
    "Expected exactly 24 distinct hourly timestamps."

assert hourly_cadence, \
    "Hourly cadence validation failed."

assert df["grid_id"].dropna().between(1, 10000).all(), \
    "Invalid grid_id detected."

assert (df[activity_columns] >= 0).all().all(), \
    "Negative activity value detected."

assert df.duplicated().sum() == 0, \
    "Exact duplicate rows detected."

print("NP1 validation PASSED")

AssertionError: Negative activity value detected.

In [15]:
canonical_columns = [
    "timestamp",
    "grid_id",
    "country_code",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity",
    "date",
    "hour",
    "day_of_week",
    "total_sms",
    "total_calls",
    "total_activity"
]

canonical_df = df[canonical_columns].copy()

display(canonical_df.head())

,timestamp,grid_id,country_code,sms_in,sms_out,call_in,call_out,internet_activity,date,hour,day_of_week,total_sms,total_calls,total_activity
0,2013-11-01,1,0,0.3521,NaN,NaN,0.0273,NaN,2013-11-01,0,Friday,0.3521,0.0273,0.3794
1,2013-11-01,1,33,NaN,NaN,NaN,NaN,0.0261,2013-11-01,0,Friday,0.0000,0.0000,0.0261
2,2013-11-01,1,39,1.7322,1.1047,0.5919,0.4020,57.7729,2013-11-01,0,Friday,2.8369,0.9939,61.6037
3,2013-11-01,2,0,0.3581,NaN,NaN,0.0273,NaN,2013-11-01,0,Friday,0.3581,0.0273,0.3854
4,2013-11-01,2,33,NaN,NaN,NaN,NaN,0.0274,2013-11-01,0,Friday,0.0000,0.0000,0.0274
